# Prática: da coleção à consulta

Nesta prática, chunks já preparados se tornam pontos em uma coleção do Qdrant. Vamos acompanhar a primeira inserção, a atualização de um ponto e uma consulta vetorial com filtro de payload.

O exemplo espera um Qdrant acessível em `http://localhost:6333`. Para usar o Qdrant Cloud, defina `QDRANT_URL` e `QDRANT_API_KEY` no ambiente. As dependências Python são `qdrant-client==1.15.1` e `sentence-transformers==5.1.0`.

## 1. Conectando ao Qdrant

A aplicação informa onde o banco está. O mesmo código funciona com o contêiner local e com uma coleção remota, sem colocar credenciais no notebook.

In [1]:
import os
import warnings
from textwrap import fill

from qdrant_client import QdrantClient, models

warnings.filterwarnings("ignore", message=".*urllib3.*")
warnings.filterwarnings("ignore", message="IProgress not found.*")

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "livros-tecnicos-pratica-artigo-05"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
server = client.info()

print(f"Qdrant conectado: {QDRANT_URL}")
print(f"versão do servidor: {server.version}")
print(f"coleção da prática: {COLLECTION_NAME}")

Qdrant conectado: http://localhost:6333
versão do servidor: 1.15.4
coleção da prática: livros-tecnicos-pratica-artigo-05


## 2. Preparando os chunks e seus payloads

Cada item já é uma unidade recuperável. O texto será transformado em vetor, enquanto os demais campos formarão o payload usado para apresentar e filtrar resultados.

O `point_id` é um inteiro aceito pelo Qdrant. O identificador editorial, como `chunk-042`, permanece explícito no payload.

In [2]:
chunks = [
    {
        "point_id": 42,
        "chunk_id": "chunk-042",
        "text": (
            "Quando um webhook expira antes da confirmação, o provedor não sabe se "
            "o evento falhou ou se apenas a resposta se perdeu. A nova tentativa deve "
            "preservar o event_id para que o consumidor reconheça a mesma entrega."
        ),
        "book_id": "integracoes-resilientes",
        "book_title": "Integrações Resilientes",
        "section": "Timeouts e confirmação",
        "pages": "118-119",
        "language": "pt-BR",
        "version": 1,
    },
    {
        "point_id": 43,
        "chunk_id": "chunk-043",
        "text": (
            "Idempotência permite repetir a entrega sem repetir seus efeitos. O consumidor "
            "persiste o event_id antes de cobrar, enviar uma notificação ou alterar estoque."
        ),
        "book_id": "integracoes-resilientes",
        "book_title": "Integrações Resilientes",
        "section": "Idempotência e duplicidade",
        "pages": "120-121",
        "language": "pt-BR",
        "version": 1,
    },
    {
        "point_id": 44,
        "chunk_id": "chunk-044",
        "text": (
            "O backoff exponencial aumenta o intervalo entre retentativas e reduz a pressão "
            "sobre um consumidor instável. Uma variação no intervalo evita novos picos."
        ),
        "book_id": "integracoes-resilientes",
        "book_title": "Integrações Resilientes",
        "section": "Retentativas e backoff",
        "pages": "122-123",
        "language": "pt-BR",
        "version": 1,
    },
    {
        "point_id": 117,
        "chunk_id": "chunk-117",
        "text": (
            "Contratos entre serviços definem formatos de requisição, autenticação, "
            "versionamento e respostas de erro que os consumidores precisam tratar."
        ),
        "book_id": "arquitetura-de-apis",
        "book_title": "Arquitetura de APIs",
        "section": "Contratos entre serviços",
        "pages": "67-68",
        "language": "pt-BR",
        "version": 1,
    },
    {
        "point_id": 281,
        "chunk_id": "chunk-281",
        "text": (
            "Alertas úteis associam um sintoma a uma ação operacional. A mensagem deve "
            "indicar o serviço afetado, a janela observada e o próximo passo esperado."
        ),
        "book_id": "operacoes-confiaveis",
        "book_title": "Operações Confiáveis",
        "section": "Alertas que exigem ação",
        "pages": "38-39",
        "language": "pt-BR",
        "version": 1,
    },
    {
        "point_id": 501,
        "chunk_id": "chunk-501",
        "text": (
            "Quando um pedido de webhook ultrapassa o tempo limite, o emissor não sabe se "
            "o processamento falhou ou se apenas a resposta se perdeu. As novas tentativas "
            "devem reutilizar o identificador do evento."
        ),
        "book_id": "entrega-fiavel-webhooks",
        "book_title": "Entrega Fiável de Webhooks",
        "section": "Tempos limite e novas tentativas",
        "pages": "42-43",
        "language": "pt-PT",
        "version": 1,
    },
]

print(f"chunks preparados: {len(chunks)}")
print("chunk      | idioma | livro")
print("-----------|--------|---------------------------")
for chunk in chunks:
    print(f"{chunk['chunk_id']:<10} | {chunk['language']:<6} | {chunk['book_title']}")

chunks preparados: 6
chunk      | idioma | livro
-----------|--------|---------------------------
chunk-042  | pt-BR  | Integrações Resilientes
chunk-043  | pt-BR  | Integrações Resilientes
chunk-044  | pt-BR  | Integrações Resilientes
chunk-117  | pt-BR  | Arquitetura de APIs
chunk-281  | pt-BR  | Operações Confiáveis
chunk-501  | pt-PT  | Entrega Fiável de Webhooks


## 3. Gerando vetores compatíveis com a coleção

Usamos o mesmo modelo dos artigos anteriores. O tamanho produzido pelo modelo definirá a dimensão de `text_dense`.

In [3]:
from sentence_transformers import SentenceTransformer
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_NAME = "text_dense"
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    [chunk["text"] for chunk in chunks],
    normalize_embeddings=True,
    show_progress_bar=False,
)
VECTOR_SIZE = embeddings.shape[1]

print(f"modelo: {MODEL_NAME}")
print(f"vetores gerados: {len(embeddings)}")
print(f"dimensões por vetor: {VECTOR_SIZE}")
print(f"espaço vetorial: {VECTOR_NAME}")

modelo: sentence-transformers/all-MiniLM-L6-v2
vetores gerados: 6
dimensões por vetor: 384
espaço vetorial: text_dense


## 4. Criando a coleção e o índice de payload

A coleção fixa a dimensão e a métrica do espaço vetorial. O índice de `language` é criado separadamente, porque armazenar um campo no payload não o transforma automaticamente em um campo indexado.

Para que a execução seja reproduzível, a célula remove apenas a coleção desta prática quando ela já existe.

In [4]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        VECTOR_NAME: models.VectorParams(
            size=VECTOR_SIZE,
            distance=models.Distance.COSINE,
        )
    },
)
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="language",
    field_schema=models.PayloadSchemaType.KEYWORD,
    wait=True,
)

collection = client.get_collection(COLLECTION_NAME)
vector_config = collection.config.params.vectors[VECTOR_NAME]

print(f"coleção criada: {COLLECTION_NAME}")
print(f"{VECTOR_NAME}: {vector_config.size} dimensões, métrica {vector_config.distance.value}")
print(f"índice de payload: language ({collection.payload_schema['language'].data_type.value})")
print(f"pontos armazenados: {collection.points_count}")

coleção criada: livros-tecnicos-pratica-artigo-05
text_dense: 384 dimensões, métrica Cosine
índice de payload: language (keyword)
pontos armazenados: 0


## 5. Inserindo os pontos pela primeira vez

O primeiro `upsert` recebe identificador, vetor e payload. Como os IDs ainda não existem, os seis pontos são inseridos.

In [5]:
def point_from_chunk(chunk, embedding):
    payload = {key: value for key, value in chunk.items() if key != "point_id"}
    return models.PointStruct(
        id=chunk["point_id"],
        vector={VECTOR_NAME: embedding.tolist()},
        payload=payload,
    )

points = [
    point_from_chunk(chunk, embedding)
    for chunk, embedding in zip(chunks, embeddings)
]

operation = client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,
)
collection = client.get_collection(COLLECTION_NAME)

print(f"operação concluída: {operation.status.value}")
print(f"pontos enviados: {len(points)}")
print(f"pontos armazenados: {collection.points_count}")

operação concluída: completed
pontos enviados: 6
pontos armazenados: 6


## 6. Atualizando o mesmo ponto com upsert

Uma nova edição amplia o trecho sobre timeout. A aplicação mantém o mesmo `point_id`, gera outro vetor e envia o payload com `version = 2`. O ponto é substituído, por isso a quantidade da coleção não aumenta.

In [6]:
updated_chunk = {
    **chunks[0],
    "text": (
        "Quando um webhook expira antes da confirmação, o status 408 indica que o "
        "consumidor não respondeu dentro do limite. Isso não prova que o evento deixou "
        "de ser processado. A retentativa deve preservar o event_id para que o consumidor "
        "reconheça a mesma entrega e evite repetir seus efeitos."
    ),
    "version": 2,
}
updated_embedding = model.encode(
    updated_chunk["text"],
    normalize_embeddings=True,
    show_progress_bar=False,
)
points_before = client.get_collection(COLLECTION_NAME).points_count

client.upsert(
    collection_name=COLLECTION_NAME,
    points=[point_from_chunk(updated_chunk, updated_embedding)],
    wait=True,
)
updated_point = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[updated_chunk["point_id"]],
    with_payload=True,
    with_vectors=False,
)[0]
points_after = client.get_collection(COLLECTION_NAME).points_count

print(f"pontos antes do upsert: {points_before}")
print(f"pontos depois do upsert: {points_after}")
print(f"ponto atualizado: {updated_point.payload['chunk_id']}")
print(f"versão armazenada: {updated_point.payload['version']}")
print("texto armazenado:")
print(fill(updated_point.payload['text'], width=72))

pontos antes do upsert: 6
pontos depois do upsert: 6
ponto atualizado: chunk-042
versão armazenada: 2
texto armazenado:
Quando um webhook expira antes da confirmação, o status 408 indica que o
consumidor não respondeu dentro do limite. Isso não prova que o evento
deixou de ser processado. A retentativa deve preservar o event_id para
que o consumidor reconheça a mesma entrega e evite repetir seus efeitos.


## 7. Consultando a coleção sem filtro

A aplicação transforma a pergunta com o mesmo modelo e envia o vetor ao espaço `text_dense`. Primeiro não restringiremos o idioma, para observar quais pontos competem pelo `limit = 3`.

In [7]:
QUERY = "como lidar com timeout em webhooks?"
query_vector = model.encode(
    QUERY,
    normalize_embeddings=True,
    show_progress_bar=False,
).tolist()

def preview(text, max_chars=150):
    normalized = " ".join(text.split())
    if len(normalized) <= max_chars:
        return normalized
    return normalized[:max_chars].rsplit(" ", 1)[0] + "..."

def print_results(title, points):
    print(title)
    for position, point in enumerate(points, start=1):
        payload = point.payload
        print(f"{position}. {payload['chunk_id']} | {payload['language']} | score={point.score:.4f}")
        print(f"   {payload['book_title']} > {payload['section']}")
        print(fill(
            preview(payload['text']),
            width=72,
            initial_indent="   ",
            subsequent_indent="   ",
        ))

unfiltered_response = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    using=VECTOR_NAME,
    limit=3,
    with_payload=True,
    with_vectors=False,
)

print(f"query: {QUERY}")
print_results("candidatos sem filtro:", unfiltered_response.points)

query: como lidar com timeout em webhooks?
candidatos sem filtro:
1. chunk-501 | pt-PT | score=0.5017
   Entrega Fiável de Webhooks > Tempos limite e novas tentativas
   Quando um pedido de webhook ultrapassa o tempo limite, o emissor não
   sabe se o processamento falhou ou se apenas a resposta se perdeu. As
   novas...
2. chunk-042 | pt-BR | score=0.4695
   Integrações Resilientes > Timeouts e confirmação
   Quando um webhook expira antes da confirmação, o status 408 indica
   que o consumidor não respondeu dentro do limite. Isso não prova que o
   evento...
3. chunk-044 | pt-BR | score=0.3518
   Integrações Resilientes > Retentativas e backoff
   O backoff exponencial aumenta o intervalo entre retentativas e reduz
   a pressão sobre um consumidor instável. Uma variação no intervalo
   evita novos...


## 8. Restringindo os candidatos com um filtro

Agora a consulta adiciona `language = pt-BR`. O filtro define quem pode participar do resultado, mas não acrescenta relevância ao score.

In [8]:
language_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="language",
            match=models.MatchValue(value="pt-BR"),
        )
    ]
)
filtered_response = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    using=VECTOR_NAME,
    query_filter=language_filter,
    limit=3,
    with_payload=True,
    with_vectors=False,
)

print_results("candidatos com language = pt-BR:", filtered_response.points)

unfiltered_ids = [point.payload["chunk_id"] for point in unfiltered_response.points]
filtered_ids = [point.payload["chunk_id"] for point in filtered_response.points]
removed_ids = [chunk_id for chunk_id in unfiltered_ids if chunk_id not in filtered_ids]
print(f"\nremovidos pelo filtro: {', '.join(removed_ids) if removed_ids else 'nenhum'}")

candidatos com language = pt-BR:
1. chunk-042 | pt-BR | score=0.4695
   Integrações Resilientes > Timeouts e confirmação
   Quando um webhook expira antes da confirmação, o status 408 indica
   que o consumidor não respondeu dentro do limite. Isso não prova que o
   evento...
2. chunk-044 | pt-BR | score=0.3518
   Integrações Resilientes > Retentativas e backoff
   O backoff exponencial aumenta o intervalo entre retentativas e reduz
   a pressão sobre um consumidor instável. Uma variação no intervalo
   evita novos...
3. chunk-117 | pt-BR | score=0.2770
   Arquitetura de APIs > Contratos entre serviços
   Contratos entre serviços definem formatos de requisição,
   autenticação, versionamento e respostas de erro que os consumidores
   precisam tratar.

removidos pelo filtro: chunk-501


## 9. Inspecionando o contrato da resposta

A consulta pediu payloads e dispensou os vetores armazenados. Isso muda o que volta para a aplicação, não o cálculo de similaridade que já produziu o ranking.

In [9]:
top_point = filtered_response.points[0]
payload_fields = ", ".join(sorted(top_point.payload))

print(f"ID interno do ponto: {top_point.id}")
print(f"identificador do chunk: {top_point.payload['chunk_id']}")
print(f"score: {top_point.score:.4f}")
print("campos do payload:")
print(fill(payload_fields, width=72, initial_indent="  ", subsequent_indent="  "))
print(f"vetor retornado: {'sim' if top_point.vector is not None else 'não'}")

ID interno do ponto: 42
identificador do chunk: chunk-042
score: 0.4695
campos do payload:
  book_id, book_title, chunk_id, language, pages, section, text, version
vetor retornado: não


A prática separou responsabilidades que podem parecer uma única operação. A aplicação definiu o espaço vetorial e o campo filtrável. O primeiro `upsert` inseriu os pontos, enquanto o segundo substituiu `chunk-042` sem aumentar a coleção. Na consulta, o vetor produziu os scores, o filtro limitou os candidatos elegíveis e `with_payload` decidiu quais dados acompanharam o resultado.

O banco devolveu candidatos próximos dentro das condições informadas. Decidir como esses candidatos serão usados continua sendo responsabilidade da aplicação.